In [1]:
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

_cwd = Path.cwd().resolve()
DATASETS_ROOT = _cwd / "Datasets" if (_cwd / "Datasets").exists() else _cwd.parent / "Datasets"
print("Datasets root:", DATASETS_ROOT)

Datasets root: /Users/tian/Downloads/Project/Datasets


# Basic Data Cleaning

Cleans **Backblaze** and **Scania** datasets for EDA and modeling.

**Pipeline** (fit on train, align val/test):
1. Drop columns with missing ratio > threshold (default 80%, subsumes All-NaN)
2. Drop constant columns (zero variance)
3. Drop duplicate columns (hash-based)

In [2]:
def find_cols_to_keep(df, exclude_cols, missing_threshold=0.8):
    """Decide which columns to keep based on training data."""
    exclude = set(exclude_cols)
    cands = [c for c in df.columns if c not in exclude]
    steps = []

    # 1. High-missing (subsumes All-NaN since ratio 1.0 > threshold)
    miss = df[cands].isna().mean()
    drop = miss[miss > missing_threshold].index.tolist()
    if drop:
        steps.append(("High-missing", drop))
        cands = [c for c in cands if c not in set(drop)]

    # 2. Constant
    nuniq = df[cands].nunique()
    drop = nuniq[nuniq <= 1].index.tolist()
    if drop:
        steps.append(("Constant", drop))
        cands = [c for c in cands if c not in set(drop)]

    # 3. Duplicate (hash-based, avoids costly transpose)
    seen, drop = {}, []
    for c in cands:
        h = hashlib.sha256(pd.util.hash_pandas_object(df[c]).values.tobytes()).hexdigest()
        if h in seen:
            drop.append(c)
        else:
            seen[h] = c
    if drop:
        steps.append(("Duplicate", drop))
        cands = [c for c in cands if c not in set(drop)]

    keep = [c for c in df.columns if c in exclude or c in set(cands)]

    for name, dropped in steps:
        preview = str(dropped[:5]) + ("..." if len(dropped) > 5 else "")
        print(f"  {name}: dropped {len(dropped)}  {preview}")
    print(f"  Result: {df.shape[1]} cols \u2192 {len(keep)} cols\n")
    return keep


def clean_split(train, val, test, exclude_cols, missing_threshold=0.8):
    """Fit on train, align val/test columns."""
    keep = find_cols_to_keep(train, exclude_cols, missing_threshold)
    return train[keep], val.reindex(columns=keep), test.reindex(columns=keep)

## 1. Backblaze

In [3]:
DATA_DIR = DATASETS_ROOT / "Backblaze" / "backblaze_data"

bb_tr = pd.read_csv(DATA_DIR / "train_set.csv", parse_dates=["date"])
bb_va = pd.read_csv(DATA_DIR / "val_set.csv", parse_dates=["date"])
bb_te = pd.read_csv(DATA_DIR / "test_set.csv", parse_dates=["date"])

print("Backblaze Cleaning:")
bb_tr, bb_va, bb_te = clean_split(
    bb_tr, bb_va, bb_te,
    exclude_cols=["date", "serial_number", "model", "capacity_bytes", "failure"],
)

bb_tr.to_csv(DATA_DIR / "train_set_cleaned.csv", index=False)
bb_va.to_csv(DATA_DIR / "val_set_cleaned.csv", index=False)
bb_te.to_csv(DATA_DIR / "test_set_cleaned.csv", index=False)
print(f"Saved. Train: {bb_tr.shape} | Val: {bb_va.shape} | Test: {bb_te.shape}")

Backblaze Cleaning:
  High-missing: dropped 126  ['smart_2_normalized', 'smart_2_raw', 'smart_8_normalized', 'smart_8_raw', 'smart_11_normalized']...
  Constant: dropped 15  ['smart_3_raw', 'smart_4_normalized', 'smart_10_normalized', 'smart_10_raw', 'smart_12_normalized']...
  Duplicate: dropped 4  ['smart_194_normalized', 'smart_194_raw', 'smart_198_normalized', 'smart_198_raw']
  Result: 179 cols → 34 cols

Saved. Train: (626293, 34) | Val: (11364, 34) | Test: (22800, 34)


## 2. Scania

In [4]:
SCANIA_DIR = DATASETS_ROOT / "SCANIA"

ops_tr = pd.read_csv(SCANIA_DIR / "train_operational_readouts.csv")
ops_va = pd.read_csv(SCANIA_DIR / "validation_operational_readouts.csv")
ops_te = pd.read_csv(SCANIA_DIR / "test_operational_readouts.csv")

print("Scania Cleaning:")
ops_tr, ops_va, ops_te = clean_split(
    ops_tr, ops_va, ops_te,
    exclude_cols=["vehicle_id", "time_step"],
)

ops_tr.to_csv(SCANIA_DIR / "train_ops_cleaned.csv", index=False)
ops_va.to_csv(SCANIA_DIR / "validation_ops_cleaned.csv", index=False)
ops_te.to_csv(SCANIA_DIR / "test_ops_cleaned.csv", index=False)
print(f"Saved. Train: {ops_tr.shape} | Val: {ops_va.shape} | Test: {ops_te.shape}")

Scania Cleaning:
  Result: 107 cols → 107 cols

Saved. Train: (1122452, 107) | Val: (119103, 107) | Test: (275264, 107)
